# LeetCode 850 — Rectangle Area II bằng Python

- Link: https://leetcode.com/problems/rectangle-area-ii/
- Độ khó: **Hard**
- Chủ đề: Hình học, Nén toạ độ, Đường quét (sweep line), Cây phân đoạn (segment tree), Bao hàm – loại trừ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def rectangleArea(self, rectangles: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho tối đa 200 hình chữ nhật song song trục toạ độ (có thể chồng lên nhau). Tính **diện tích phần hợp** của chúng (vùng bị phủ bởi ít nhất một hình), lấy modulo `10⁹ + 7`.

Toạ độ tới `10⁹` nên không thể "tô từng ô". Có hai ý tưởng đẹp:

- **Nén toạ độ:** chỉ có tối đa `2n` giá trị `x` và `2n` giá trị `y` "đặc biệt"; chia mặt phẳng thành lưới `≤ 400 × 400` ô chữ nhật không đều, mỗi ô hoặc bị phủ hoàn toàn hoặc không.
- **Đường quét:** quét một đường thẳng đứng từ trái sang phải; giữa hai sự kiện liên tiếp, diện tích tăng thêm `(độ dài phần trục y bị phủ) × (khoảng cách x)`. Độ dài phủ được duy trì bằng danh sách khoảng hoặc **cây phân đoạn**.

Sau notebook này bạn sẽ:

1. Tính diện tích hợp bằng brute force trên lưới đơn vị (để đối chiếu).
2. Nén toạ độ và đánh dấu lưới ô — `O(n³)`.
3. Cài đường quét với sự kiện "mở / đóng" và hợp các khoảng — `O(n² log n)`.
4. Cài cây phân đoạn đếm phủ (`count`, `covered length`) — `O(n log n)`.
5. Hiểu bao hàm – loại trừ và vì sao nó chỉ dùng được với `n` rất nhỏ.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `rectangles`, `rectangles[i] = [x1, y1, x2, y2]` với `(x1, y1)` là góc dưới trái và `(x2, y2)` là góc trên phải của hình chữ nhật thứ `i` (song song trục). Tính tổng diện tích vùng bị phủ bởi ít nhất một hình chữ nhật. Phần chồng lên nhau chỉ tính một lần. Trả về kết quả modulo `10⁹ + 7`.

**Ví dụ 1**

```text
Input: rectangles = [[0,0,2,2],[1,0,2,3],[1,0,3,1]]
Output: 6
Giải thích: tổng diện tích phần hợp là 6.
```

**Ví dụ 2**

```text
Input: rectangles = [[0,0,1000000000,1000000000]]
Output: 49
Giải thích: 10¹⁸ mod (10⁹ + 7) = 49.
```

**Ràng buộc**

- `1 <= rectangles.length <= 200`
- `rectangles[i].length == 4`
- `0 <= x1, y1, x2, y2 <= 10⁹`
- `x1 <= x2`, `y1 <= y2` (có thể có hình chữ nhật diện tích 0)

In [1]:
from typing import List

MOD = 10 ** 9 + 7

EXAMPLES = [
    ([[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]], 6),
    ([[0, 0, 1000000000, 1000000000]], 49),
    ([[0, 0, 1, 1], [2, 2, 3, 3]], 2),                 # rời nhau
    ([[0, 0, 3, 3], [1, 1, 2, 2]], 9),                 # lồng nhau
    ([[0, 0, 0, 5], [1, 1, 3, 3]], 4),                 # hình diện tích 0
]
for r, exp in EXAMPLES:
    print(r, "-> kỳ vọng", exp)

[[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]] -> kỳ vọng 6
[[0, 0, 1000000000, 1000000000]] -> kỳ vọng 49
[[0, 0, 1, 1], [2, 2, 3, 3]] -> kỳ vọng 2
[[0, 0, 3, 3], [1, 1, 2, 2]] -> kỳ vọng 9
[[0, 0, 0, 5], [1, 1, 3, 3]] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Nén toạ độ

Lấy tập các toạ độ, sắp xếp, gán chỉ số. `index[x]` cho biết `x` là giá trị thứ mấy.

In [2]:
xs = sorted({0, 2, 1, 2, 1, 3})
index = {x: i for i, x in enumerate(xs)}
print(xs, index)

[0, 1, 2, 3] {0: 0, 1: 1, 2: 2, 3: 3}


### 3.2. Hợp các khoảng trên một trục

Sắp xếp các khoảng `[a, b)` theo `a`, gộp khi chồng nhau, cộng độ dài.

In [3]:
def union_length(intervals):
    total = 0
    cur_a = cur_b = None
    for a, b in sorted(intervals):
        if cur_b is None or a > cur_b:
            if cur_b is not None:
                total += cur_b - cur_a
            cur_a, cur_b = a, b
        else:
            cur_b = max(cur_b, b)
    if cur_b is not None:
        total += cur_b - cur_a
    return total

print(union_length([(0, 2), (1, 3), (5, 6)]))      # [0,3) ∪ [5,6) = 4

4


### 3.3. Sự kiện cho đường quét

Mỗi hình chữ nhật tạo hai sự kiện: tại `x1` "mở" khoảng `[y1, y2)`, tại `x2` "đóng" khoảng đó. Sắp xếp sự kiện theo `x`.

In [4]:
rects = [[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]]
events = []
for x1, y1, x2, y2 in rects:
    events.append((x1, +1, y1, y2))
    events.append((x2, -1, y1, y2))
events.sort()
for e in events:
    print(e)

(0, 1, 0, 2)
(1, 1, 0, 1)
(1, 1, 0, 3)
(2, -1, 0, 2)
(2, -1, 0, 3)
(3, -1, 0, 1)


### 3.4. Modulo với số nguyên lớn

Python có số nguyên không giới hạn, nên có thể cộng hết rồi `% MOD` một lần ở cuối. Ở ngôn ngữ khác (C++/Java) phải lấy mod dần và dùng kiểu 64 bit.

In [5]:
print((10 ** 9) * (10 ** 9) % MOD)

49


### 3.5. Cây phân đoạn (segment tree) — ý tưởng

Cây nhị phân trên các đoạn sơ cấp `[ys[i], ys[i+1])`. Mỗi nút giữ:

- `cnt[node]`: số hình chữ nhật đang phủ **toàn bộ** đoạn của nút (không đẩy xuống con).
- `length[node]`: tổng độ dài bị phủ trong đoạn của nút.

`length[node] = độ dài đoạn` nếu `cnt > 0`, ngược lại = tổng `length` hai con (hoặc 0 nếu là lá).

## 4. Ý tưởng nền tảng / trực giác

### 4.1. Ví dụ 1 trên giấy

```text
y
3 |   B
2 | A A/B
1 | A A/B
0 | A/C A/B/C C
  +---+---+---+ x
  0   1   2   3
```

- Ô `[0,1)×[0,2)`: phủ (A) → 2
- Ô `[1,2)×[0,3)`: phủ (A/B/C) → 3
- Ô `[2,3)×[0,1)`: phủ (C) → 1

Tổng `2 + 3 + 1 = 6`.

### 4.2. Lưới nén

`xs = [0, 1, 2, 3]`, `ys = [0, 1, 2, 3]` → 3 × 3 ô chữ nhật. Mỗi ô `[xs[i], xs[i+1]) × [ys[j], ys[j+1])` hoặc nằm trọn trong một hình, hoặc không giao phần trong với hình nào — vì không có cạnh hình nào cắt ngang qua ô.

### 4.3. Đường quét

| đoạn x | khoảng y đang mở | độ dài phủ | diện tích thêm |
|---|---|---|---|
| [0,1) | A: [0,2) | 2 | 2·1 = 2 |
| [1,2) | A [0,2), B [0,3), C [0,1) | 3 | 3·1 = 3 |
| [2,3) | C [0,1) | 1 | 1·1 = 1 |

Tổng 6.

In [6]:
def draw(rects, W, H):
    for y in range(H - 1, -1, -1):
        row = ""
        for x in range(W):
            names = "".join(chr(65 + i) for i, (x1, y1, x2, y2) in enumerate(rects) if x1 <= x < x2 and y1 <= y < y2)
            row += f"{names or '.':>4}"
        print(f"{y} |{row}")
    print("   " + "".join(f"{x:>4}" for x in range(W)))

draw(rects, 3, 3)

2 |   .   B   .
1 |   A  AB   .
0 |   A ABC   C
      0   1   2


## 5. Cách 1 — Brute Force: đếm từng ô đơn vị

**Ý tưởng.** (Chỉ khi toạ độ nhỏ.) Với mỗi ô `1×1` `[x, x+1) × [y, y+1)`, kiểm tra có hình nào phủ không.

In [7]:
def area_brute(rects):
    cells = set()
    for x1, y1, x2, y2 in rects:
        for x in range(x1, x2):
            for y in range(y1, y2):
                cells.add((x, y))
    return len(cells) % MOD

for r, exp in EXAMPLES:
    if max(max(t) for t in r) <= 100:
        assert area_brute(r) == exp
print("Brute force (toạ độ nhỏ): OK")

Brute force (toạ độ nhỏ): OK


**Độ phức tạp.** `O(tổng diện tích)` — với toạ độ `10⁹` là `10¹⁸` ô. Hoàn toàn không khả thi.

**Nút thắt.** Hầu hết các ô đơn vị "giống nhau": cả một vùng lớn không có cạnh hình nào đi qua đều bị phủ hoặc không phủ như nhau. Chỉ những toạ độ là **cạnh của hình** mới quan trọng → nén toạ độ.

## 6. Bước đột phá

**Insight 1 — Chỉ `2n` toạ độ quan trọng mỗi trục.** Giữa hai giá trị `x` liên tiếp trong tập `{x1, x2}` của mọi hình, và hai `y` liên tiếp, ô chữ nhật không bị cạnh nào cắt → đồng nhất. Lưới nén có `≤ (2n − 1)²` ô.

**Insight 2 — Đường quét: diện tích = ∫ (độ dài phủ trên trục y) dx.** Độ dài phủ là hàm bậc thang chỉ đổi tại các `x` sự kiện. Giữa hai sự kiện, diện tích tăng `L · Δx`.

**Insight 3 — Cây phân đoạn cho cập nhật nhanh.** Mỗi sự kiện là "cộng/trừ 1 trên đoạn `[y1, y2)`", truy vấn là "tổng độ dài các đoạn có đếm > 0". Kỹ thuật đặc biệt: **không cần đẩy lười** vì mỗi lần trừ luôn khớp đúng một lần cộng trước đó trên cùng đoạn → `cnt` ở nút không bao giờ âm.

**Quy tắc nếu…thì:**
- *Nếu* `x1 == x2` hoặc `y1 == y2` *thì* hình diện tích 0, không ảnh hưởng (bỏ qua được).
- *Nếu* hai sự kiện cùng `x` *thì* `Δx = 0`, thứ tự xử lý giữa chúng không ảnh hưởng diện tích.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Nén toạ độ + đánh dấu lưới, `O(n³)`

In [8]:
def area_grid(rects):
    xs = sorted({x for r in rects for x in (r[0], r[2])})
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    xi = {x: i for i, x in enumerate(xs)}
    yi = {y: i for i, y in enumerate(ys)}
    grid = [[False] * len(ys) for _ in range(len(xs))]
    for x1, y1, x2, y2 in rects:
        for i in range(xi[x1], xi[x2]):
            for j in range(yi[y1], yi[y2]):
                grid[i][j] = True
    total = 0
    for i in range(len(xs) - 1):
        for j in range(len(ys) - 1):
            if grid[i][j]:
                total += (xs[i + 1] - xs[i]) * (ys[j + 1] - ys[j])
    return total % MOD

for r, exp in EXAMPLES:
    assert area_grid(r) == exp
print("Nén toạ độ + lưới: OK")

Nén toạ độ + lưới: OK


Giải thích:

- `grid[i][j]` ứng với ô `[xs[i], xs[i+1]) × [ys[j], ys[j+1])`.
- Hình `[x1, x2) × [y1, y2)` phủ các ô `i ∈ [xi[x1], xi[x2])`, `j ∈ [yi[y1], yi[y2])` — nửa mở, nên ô bên phải `x2` không bị đánh dấu.

Code nộp LeetCode cho cách 2:

In [9]:
class SolutionGrid:
    def rectangleArea(self, rectangles: List[List[int]]) -> int:
        xs = sorted({x for r in rectangles for x in (r[0], r[2])})
        ys = sorted({y for r in rectangles for y in (r[1], r[3])})
        xi = {x: i for i, x in enumerate(xs)}
        yi = {y: i for i, y in enumerate(ys)}
        grid = [[0] * len(ys) for _ in range(len(xs))]
        for x1, y1, x2, y2 in rectangles:
            for i in range(xi[x1], xi[x2]):
                row = grid[i]
                for j in range(yi[y1], yi[y2]):
                    row[j] = 1
        total = 0
        for i in range(len(xs) - 1):
            w = xs[i + 1] - xs[i]
            row = grid[i]
            for j in range(len(ys) - 1):
                if row[j]:
                    total += w * (ys[j + 1] - ys[j])
        return total % (10 ** 9 + 7)

for r, exp in EXAMPLES:
    assert SolutionGrid().rectangleArea(r) == exp
print("SolutionGrid: OK")

SolutionGrid: OK


**Độ phức tạp cách 2:** mỗi hình đánh dấu tối đa `(2n)²` ô → `O(n³)` = `200 · 400² = 3.2·10⁷` xấu nhất (Python vài giây, có thể chậm trên LeetCode); bộ nhớ `O(n²)`.

### 7.2. Cách 3 — Đường quét + hợp khoảng, `O(n² log n)`

Duyệt sự kiện theo `x`. Trước khi xử lý sự kiện tại `x`, cộng `union_length(active) · (x − prev_x)`. Sau đó thêm/xoá khoảng.

In [10]:
def area_sweep(rects):
    events = []
    for x1, y1, x2, y2 in rects:
        if x1 < x2 and y1 < y2:
            events.append((x1, 1, y1, y2))
            events.append((x2, -1, y1, y2))
    events.sort()
    active = []                      # danh sách (y1, y2) đang mở (có thể trùng)
    total = 0
    prev_x = None
    for x, typ, y1, y2 in events:
        if prev_x is not None and x > prev_x:
            total += union_length(active) * (x - prev_x)
        if typ == 1:
            active.append((y1, y2))
        else:
            active.remove((y1, y2))
        prev_x = x
    return total % MOD

for r, exp in EXAMPLES:
    assert area_sweep(r) == exp
print("Đường quét + hợp khoảng: OK")

Đường quét + hợp khoảng: OK


Code nộp LeetCode cho cách 3:

In [11]:
class SolutionSweep:
    def rectangleArea(self, rectangles: List[List[int]]) -> int:
        events = []
        for x1, y1, x2, y2 in rectangles:
            if x1 < x2 and y1 < y2:                # bỏ hình diện tích 0
                events.append((x1, 1, y1, y2))
                events.append((x2, -1, y1, y2))
        if not events:
            return 0
        events.sort()
        active = []
        total, prev_x = 0, events[0][0]
        for x, typ, y1, y2 in events:
            if x > prev_x and active:
                covered, cur = 0, -1
                for a, b in sorted(active):        # hợp các khoảng
                    if b > cur:
                        covered += b - max(a, cur)
                        cur = b
                total += covered * (x - prev_x)
            if typ == 1:
                active.append((y1, y2))
            else:
                active.remove((y1, y2))
            prev_x = x
        return total % (10 ** 9 + 7)

for r, exp in EXAMPLES:
    assert SolutionSweep().rectangleArea(r) == exp
print("SolutionSweep: OK")

SolutionSweep: OK


Vì sao phải bỏ hình diện tích 0? Với `x1 == x2`, sự kiện đóng `(x, −1, …)` được sắp **trước** sự kiện mở `(x, +1, …)` → `remove` một khoảng chưa có → lỗi. Bỏ chúng là an toàn vì chúng không đóng góp diện tích.

Mẹo hợp khoảng gọn: duy trì `cur` = điểm phải nhất đã phủ; với khoảng `[a, b)` đã sắp, phần mới là `b − max(a, cur)` nếu `b > cur`.

**Độ phức tạp cách 3:** `2n` sự kiện, mỗi lần sắp `O(n log n)` → `O(n² log n)`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — Đường quét + cây phân đoạn, `O(n log n)`

Nén trục `y` thành `ys`. Đoạn sơ cấp `k` là `[ys[k], ys[k+1])`. Cây quản lý `m = len(ys) − 1` đoạn.

In [12]:
class CoverTree:
    def __init__(self, ys):
        self.ys = ys
        self.m = len(ys) - 1
        self.cnt = [0] * (4 * max(1, self.m))
        self.length = [0] * (4 * max(1, self.m))

    def update(self, l, r, val, node=1, lo=0, hi=None):
        """Cộng val lên các đoạn sơ cấp [l, r) (chỉ số)."""
        if hi is None:
            hi = self.m
        if r <= lo or hi <= l:
            return
        if l <= lo and hi <= r:
            self.cnt[node] += val
        else:
            mid = (lo + hi) // 2
            self.update(l, r, val, 2 * node, lo, mid)
            self.update(l, r, val, 2 * node + 1, mid, hi)
        # tính lại độ dài phủ của nút
        if self.cnt[node] > 0:
            self.length[node] = self.ys[hi] - self.ys[lo]
        elif hi - lo == 1:
            self.length[node] = 0
        else:
            self.length[node] = self.length[2 * node] + self.length[2 * node + 1]

    def covered(self):
        return self.length[1]

def area_segtree(rects):
    rects = [r for r in rects if r[0] < r[2] and r[1] < r[3]]
    if not rects:
        return 0
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    yi = {y: i for i, y in enumerate(ys)}
    events = []
    for x1, y1, x2, y2 in rects:
        events.append((x1, 1, yi[y1], yi[y2]))
        events.append((x2, -1, yi[y1], yi[y2]))
    events.sort()
    tree = CoverTree(ys)
    total, prev_x = 0, events[0][0]
    for x, val, l, r in events:
        total += tree.covered() * (x - prev_x)
        tree.update(l, r, val)
        prev_x = x
    return total % MOD

for r, exp in EXAMPLES:
    assert area_segtree(r) == exp
print("Đường quét + cây phân đoạn: OK")

Đường quét + cây phân đoạn: OK


Vì sao không cần đẩy lười (lazy propagation)? Một lần `update(l, r, +1)` tăng `cnt` của **cùng một tập nút** (phân rã chuẩn của `[l, r)`) mà `update(l, r, −1)` về sau sẽ giảm. Do đó `cnt` mỗi nút luôn ≥ 0 và `length` được tính đúng từ `cnt` của chính nút và `length` của con.

Code nộp LeetCode cho cách 4:

In [13]:
class SolutionSegTree:
    def rectangleArea(self, rectangles: List[List[int]]) -> int:
        ys = sorted({y for r in rectangles for y in (r[1], r[3])})
        yi = {y: i for i, y in enumerate(ys)}
        m = len(ys) - 1
        if m <= 0:
            return 0
        cnt = [0] * (4 * m)
        length = [0] * (4 * m)

        def update(node, lo, hi, l, r, val):
            if r <= lo or hi <= l:
                return
            if l <= lo and hi <= r:
                cnt[node] += val
            else:
                mid = (lo + hi) // 2
                update(2 * node, lo, mid, l, r, val)
                update(2 * node + 1, mid, hi, l, r, val)
            if cnt[node]:
                length[node] = ys[hi] - ys[lo]
            elif hi - lo == 1:
                length[node] = 0
            else:
                length[node] = length[2 * node] + length[2 * node + 1]

        events = []
        for x1, y1, x2, y2 in rectangles:
            if x1 < x2 and y1 < y2:
                events.append((x1, 1, yi[y1], yi[y2]))
                events.append((x2, -1, yi[y1], yi[y2]))
        events.sort()
        total, prev_x = 0, events[0][0] if events else 0
        for x, val, l, r in events:
            total += length[1] * (x - prev_x)
            update(1, 0, m, l, r, val)
            prev_x = x
        return total % (10 ** 9 + 7)

for r, exp in EXAMPLES:
    assert SolutionSegTree().rectangleArea(r) == exp
print("SolutionSegTree: OK")

SolutionSegTree: OK


**Độ phức tạp cách 4:** `O(n log n)` (sắp xếp + `2n` cập nhật `O(log n)`); bộ nhớ `O(n)`.

### 7.4. Cách 5 — Bao hàm – loại trừ (chỉ cho `n` rất nhỏ)

`|A₁ ∪ ... ∪ Aₙ| = Σ|Aᵢ| − Σ|Aᵢ ∩ Aⱼ| + Σ|Aᵢ ∩ Aⱼ ∩ Aₖ| − ...`. Giao của các hình chữ nhật song song trục vẫn là hình chữ nhật (hoặc rỗng). Duyệt mọi tập con: `O(2ⁿ · n)`.

In [14]:
def area_incl_excl(rects):
    n = len(rects)
    total = 0
    for mask in range(1, 1 << n):
        x1 = y1 = -1
        x2 = y2 = float('inf')
        bits = 0
        for i in range(n):
            if mask >> i & 1:
                bits += 1
                a, b, c, d = rects[i]
                x1, y1, x2, y2 = max(x1, a), max(y1, b), min(x2, c), min(y2, d)
        if x1 < x2 and y1 < y2:
            area = (x2 - x1) * (y2 - y1)
            total += area if bits % 2 else -area
    return total % MOD

for r, exp in EXAMPLES:
    assert area_incl_excl(r) == exp
print("Bao hàm – loại trừ: OK")

Bao hàm – loại trừ: OK


Có thể cắt tỉa: khi giao đã rỗng thì mọi tập lớn hơn chứa nó cũng rỗng (duyệt DFS). Nhưng xấu nhất (mọi hình chồng nhau) vẫn `2ⁿ` — không dùng được với `n = 200`.

## 8. TRACE từng bước

Trace cách 3 (đường quét + hợp khoảng) trên ví dụ 1.

In [15]:
def trace_sweep(rects):
    events = []
    for idx, (x1, y1, x2, y2) in enumerate(rects):
        events.append((x1, 1, y1, y2, chr(65 + idx)))
        events.append((x2, -1, y1, y2, chr(65 + idx)))
    events.sort()
    active, total, prev_x = [], 0, events[0][0]
    for x, typ, y1, y2, name in events:
        if x > prev_x:
            L = union_length([(a, b) for a, b, _ in active])
            total += L * (x - prev_x)
            print(f"  đoạn x [{prev_x},{x}): khoảng mở {[(a, b, nm) for a, b, nm in active]} → phủ {L} × {x - prev_x} = {L * (x - prev_x)}; tổng = {total}")
        if typ == 1:
            active.append((y1, y2, name))
            print(f"x={x}: MỞ  {name} [{y1},{y2})")
        else:
            active.remove((y1, y2, name))
            print(f"x={x}: ĐÓNG {name} [{y1},{y2})")
        prev_x = x
    print("Diện tích =", total)
    return total

trace_sweep([[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]])

x=0: MỞ  A [0,2)
  đoạn x [0,1): khoảng mở [(0, 2, 'A')] → phủ 2 × 1 = 2; tổng = 2
x=1: MỞ  C [0,1)
x=1: MỞ  B [0,3)
  đoạn x [1,2): khoảng mở [(0, 2, 'A'), (0, 1, 'C'), (0, 3, 'B')] → phủ 3 × 1 = 3; tổng = 5
x=2: ĐÓNG A [0,2)
x=2: ĐÓNG B [0,3)
  đoạn x [2,3): khoảng mở [(0, 1, 'C')] → phủ 1 × 1 = 1; tổng = 6
x=3: ĐÓNG C [0,1)
Diện tích = 6


6

Trace cách 4 — in `length[1]` (độ dài phủ toàn trục y) sau mỗi cập nhật:

In [16]:
def trace_segtree(rects):
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    yi = {y: i for i, y in enumerate(ys)}
    print("ys =", ys, "→ đoạn sơ cấp:", [(ys[k], ys[k + 1]) for k in range(len(ys) - 1)])
    events = sorted([(r[0], 1, yi[r[1]], yi[r[3]]) for r in rects] + [(r[2], -1, yi[r[1]], yi[r[3]]) for r in rects])
    tree = CoverTree(ys)
    total, prev_x = 0, events[0][0]
    for x, val, l, r in events:
        add = tree.covered() * (x - prev_x)
        total += add
        tree.update(l, r, val)
        print(f"x={x:>2}: cộng {add:>2} (tổng {total:>2}); {'+1' if val > 0 else '-1'} trên đoạn chỉ số [{l},{r}) → phủ hiện tại = {tree.covered()}")
        prev_x = x
    return total

trace_segtree([[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]])

ys = [0, 1, 2, 3] → đoạn sơ cấp: [(0, 1), (1, 2), (2, 3)]
x= 0: cộng  0 (tổng  0); +1 trên đoạn chỉ số [0,2) → phủ hiện tại = 2
x= 1: cộng  2 (tổng  2); +1 trên đoạn chỉ số [0,1) → phủ hiện tại = 2
x= 1: cộng  0 (tổng  2); +1 trên đoạn chỉ số [0,3) → phủ hiện tại = 3
x= 2: cộng  3 (tổng  5); -1 trên đoạn chỉ số [0,2) → phủ hiện tại = 3
x= 2: cộng  0 (tổng  5); -1 trên đoạn chỉ số [0,3) → phủ hiện tại = 1
x= 3: cộng  1 (tổng  6); -1 trên đoạn chỉ số [0,1) → phủ hiện tại = 0


6

Bảng trạng thái (cách 3):

| x sự kiện | trước sự kiện: phủ × Δx | sau sự kiện: tập mở |
|---|---|---|
| 0 | — | {A} |
| 1 | 2 × 1 = 2 | {A, B, C} |
| 2 | 3 × 1 = 3 (tổng 5) | {C} (đóng A, B) |
| 3 | 1 × 1 = 1 (tổng 6) | {} |

## 9. Code nộp LeetCode

Chọn cách 4 (đường quét + cây phân đoạn): `O(n log n)`, kỹ thuật chuẩn cho bài diện tích hợp.

In [17]:
from typing import List

class Solution:
    def rectangleArea(self, rectangles: List[List[int]]) -> int:
        MOD = 10 ** 9 + 7
        ys = sorted({y for r in rectangles for y in (r[1], r[3])})
        yi = {y: i for i, y in enumerate(ys)}
        m = len(ys) - 1                         # số đoạn sơ cấp trên trục y
        if m <= 0:
            return 0
        cnt = [0] * (4 * m)                     # số hình phủ trọn đoạn của nút
        length = [0] * (4 * m)                  # độ dài bị phủ trong đoạn của nút

        def update(node, lo, hi, l, r, val):
            if r <= lo or hi <= l:
                return
            if l <= lo and hi <= r:
                cnt[node] += val
            else:
                mid = (lo + hi) // 2
                update(2 * node, lo, mid, l, r, val)
                update(2 * node + 1, mid, hi, l, r, val)
            if cnt[node] > 0:
                length[node] = ys[hi] - ys[lo]
            elif hi - lo == 1:
                length[node] = 0
            else:
                length[node] = length[2 * node] + length[2 * node + 1]

        events = []
        for x1, y1, x2, y2 in rectangles:
            if x1 < x2 and y1 < y2:
                events.append((x1, 1, yi[y1], yi[y2]))    # mở
                events.append((x2, -1, yi[y1], yi[y2]))   # đóng
        if not events:
            return 0
        events.sort()
        total, prev_x = 0, events[0][0]
        for x, val, l, r in events:
            total += length[1] * (x - prev_x)              # diện tích dải [prev_x, x)
            update(1, 0, m, l, r, val)
            prev_x = x
        return total % MOD

for r, exp in EXAMPLES:
    assert Solution().rectangleArea(r) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Nén toạ độ (cách 2).** Gọi `xs`, `ys` là các toạ độ cạnh đã sắp xếp. Xét ô `Q = [xs[i], xs[i+1]) × [ys[j], ys[j+1])`. Với mỗi hình `R = [x1, x2) × [y1, y2)`, vì `x1, x2 ∈ xs` và giữa `xs[i]`, `xs[i+1]` không có giá trị nào của `xs`, nên hoặc `[xs[i], xs[i+1]) ⊆ [x1, x2)` hoặc hai khoảng rời nhau; tương tự với `y`. Vậy `Q ⊆ R` hoặc `Q ∩ R = ∅`. Các ô phân hoạch vùng bao, nên diện tích hợp = tổng diện tích các ô bị phủ bởi ít nhất một hình.

**Đường quét (cách 3, 4).** Gọi `L(x)` = độ dài (độ đo) của tập `{y : (x, y) thuộc hợp}`. Theo nguyên lý Cavalieri, diện tích = `∫ L(x) dx`. `L` là hằng số trên mỗi khoảng `[xₖ, xₖ₊₁)` giữa hai sự kiện liên tiếp, và bằng độ dài hợp các khoảng `[y1, y2)` của các hình có `x1 <= xₖ < x2` — chính là tập `active` sau khi xử lý mọi sự kiện tại `xₖ`. Vậy diện tích = `Σ L(xₖ)·(xₖ₊₁ − xₖ)`. (Các sự kiện cùng `x` có `Δx = 0` nên thứ tự giữa chúng không ảnh hưởng.)

**Cây phân đoạn.** Bất biến: `length[node]` = độ đo của phần đoạn của nút bị phủ bởi ít nhất một khoảng **có trong phân rã đi qua nút hoặc con cháu của nó**. Nếu `cnt[node] > 0`, cả đoạn bị phủ. Nếu không, phần phủ là hợp phần phủ của hai con. Vì mỗi khoảng `[l, r)` được thêm và gỡ trên **cùng** tập nút phân rã, `cnt` luôn ≥ 0 và bất biến được giữ sau mỗi cập nhật (tính lại từ dưới lên dọc đường đi). Gốc cho `L(x)`.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Ô đơn vị | `O(tổng diện tích)` | `O(diện tích)` |
| 2. Nén + lưới | `O(n³)` | `O(n²)` |
| 3. Quét + hợp khoảng | `O(n² log n)` | `O(n)` |
| 4. Quét + cây phân đoạn | `O(n log n)` | `O(n)` |
| 5. Bao hàm – loại trừ | `O(2ⁿ · n)` | `O(1)` |

In [18]:
import random, time

def random_rects(n, C, rnd):
    res = []
    for _ in range(n):
        x1, x2 = sorted(rnd.randint(0, C) for _ in range(2))
        y1, y2 = sorted(rnd.randint(0, C) for _ in range(2))
        res.append([x1, y1, x2, y2])
    return res

rnd = random.Random(850)
big = random_rects(200, 10 ** 9, rnd)
for name, fn in [("grid", area_grid), ("sweep", area_sweep), ("segtree", area_segtree),
                 ("Solution", lambda r: Solution().rectangleArea(r))]:
    t0 = time.perf_counter()
    res = fn(big)
    print(f"{name:9}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

grid     : 40350723  (143 ms)


sweep    : 40350723  (7 ms)
segtree  : 40350723  (4 ms)


Solution : 40350723  (5 ms)


## 12. Bộ test

In [19]:
methods = {
    "brute": area_brute,
    "grid": area_grid,
    "sweep": area_sweep,
    "segtree": area_segtree,
    "incl_excl": area_incl_excl,
    "SolutionGrid": lambda r: SolutionGrid().rectangleArea(r),
    "SolutionSweep": lambda r: SolutionSweep().rectangleArea(r),
    "SolutionSegTree": lambda r: SolutionSegTree().rectangleArea(r),
    "Solution": lambda r: Solution().rectangleArea(r),
}

# 12.1 Ví dụ + edge cases (brute chỉ chạy khi toạ độ nhỏ)
edge = EXAMPLES + [
    ([[0, 0, 1, 1]] * 3, 1),                          # trùng hoàn toàn
    ([[0, 0, 2, 1], [1, 0, 3, 1]], 3),                 # chồng một phần theo x
    ([[0, 0, 0, 0]], 0),                               # chỉ có hình diện tích 0
    ([[0, 0, 10, 1], [0, 0, 1, 10]], 19),              # chữ L
]
for name, fn in methods.items():
    for r, exp in edge:
        if name == "brute" and max(max(t) for t in r) > 100:
            continue
        assert fn(r) == exp, (name, r, fn(r), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed, toạ độ nhỏ: đối chiếu brute force
for seed in range(400):
    r_ = random.Random(seed)
    rects = random_rects(r_.randint(1, 7), r_.randint(1, 12), r_)
    ref = area_brute(rects)
    for name, fn in methods.items():
        assert fn(rects) == ref, (name, rects)
print("400 bộ ngẫu nhiên: khớp brute force")

# 12.3 Toạ độ lớn: các cách nhanh khớp nhau (và khớp bao hàm–loại trừ khi n nhỏ)
for seed in range(30):
    r_ = random.Random(1000 + seed)
    rects = random_rects(r_.randint(1, 10), 10 ** 9, r_)
    vals = {name: fn(rects) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("Toạ độ lớn: các cách khớp nhau")

Ví dụ + edge cases: OK


400 bộ ngẫu nhiên: khớp brute force
Toạ độ lớn: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cộng diện tích từng hình (quên phần chồng)

**Code sai:**

In [20]:
def wrong_sum_areas(rects):
    return sum((x2 - x1) * (y2 - y1) for x1, y1, x2, y2 in rects) % MOD   # SAI

print("sai:", wrong_sum_areas(EXAMPLES[0][0]), " đúng:", EXAMPLES[0][1])

sai: 9  đúng: 6


**Vì sao sai:** phần chồng bị tính nhiều lần (`4 + 3 + 2 = 9` thay vì 6). **Code đúng:** tính diện tích **hợp**.

### Lỗi 2 — Quên lấy modulo (hoặc lấy modulo trên toạ độ)

**Code sai:**

In [21]:
def wrong_no_mod(rects):
    return area_grid_raw(rects)                            # SAI: không % MOD

def wrong_mod_coords(rects):
    rects = [[v % 1000 for v in r] for r in rects]         # SAI: "thu nhỏ" toạ độ bằng mod
    return area_grid(rects)

def area_grid_raw(rects):
    xs = sorted({x for r in rects for x in (r[0], r[2])})
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    total = 0
    for i in range(len(xs) - 1):
        for j in range(len(ys) - 1):
            if any(r[0] <= xs[i] < r[2] and r[1] <= ys[j] < r[3] for r in rects):
                total += (xs[i + 1] - xs[i]) * (ys[j + 1] - ys[j])
    return total

big1 = [[0, 0, 10 ** 9, 10 ** 9]]
print("không mod:", wrong_no_mod(big1), "| mod trên toạ độ:", wrong_mod_coords([[0, 0, 1500, 1500]]),
      "| đúng:", Solution().rectangleArea(big1), "và", Solution().rectangleArea([[0, 0, 1500, 1500]]))

không mod: 1000000000000000000 | mod trên toạ độ: 250000 | đúng: 49 và 2250000


**Vì sao sai:** đề yêu cầu kết quả modulo `10⁹ + 7`; Python không tràn số nên quên mod vẫn chạy nhưng sai đáp án. Còn lấy mod trên **toạ độ** làm méo hình (`1500 % 1000 = 500`). Nguyên tắc: chỉ lấy mod trên **kết quả cộng/nhân**, không trên dữ liệu hình học và không trộn với phép so sánh `min/max`.

### Lỗi 3 — Hợp khoảng sai khi các khoảng chạm nhau / lồng nhau

**Code sai:**

In [22]:
def wrong_union(intervals):
    total = 0
    prev_end = -1
    for a, b in sorted(intervals):
        if a >= prev_end:
            total += b - a
        else:
            total += b - prev_end              # SAI: khi b < prev_end (lồng nhau) cộng số âm
        prev_end = b                           # SAI: phải là max(prev_end, b)
    return total

print("sai:", wrong_union([(0, 10), (2, 3), (4, 5)]), " đúng:", union_length([(0, 10), (2, 3), (4, 5)]))

sai: 4  đúng: 10


**Vì sao sai:** khoảng `(2, 3)` nằm trong `(0, 10)` → cộng `3 − 10 = −7`, rồi `prev_end` bị kéo lùi về 3. **Code đúng:** chỉ cộng khi `b > cur`, cộng `b − max(a, cur)`, cập nhật `cur = max(cur, b)`.

### Lỗi 4 — Cây phân đoạn dùng đỉnh `ys[i]` làm lá thay vì đoạn `[ys[i], ys[i+1])`

Nếu lá là điểm `ys[i]` thì độ dài lá bằng 0, hoặc phải cộng `+1` kiểu "đếm điểm nguyên" — sai với toạ độ liên tục. Lá phải là **đoạn sơ cấp** và `update` nhận khoảng chỉ số nửa mở `[yi[y1], yi[y2])`.

## 14. Mẹo & nhận dạng pattern

- **"Diện tích / chu vi / độ dài hợp" của các hình song song trục** → đường quét + cây phân đoạn đếm phủ (không lazy).
- **Toạ độ lớn, số lượng nhỏ** → **nén toạ độ**.
- **Đường quét** = biến bài 2D thành chuỗi bài 1D (hợp khoảng) theo thời gian `x`.
- **Hợp khoảng 1D**: sắp theo đầu trái, giữ `cur` = đầu phải xa nhất.
- Bài liên quan: **223 Rectangle Area** (2 hình), **391 Perfect Rectangle**, **218 The Skyline Problem**, **56 Merge Intervals**, **2276 Count Integers in Intervals**.

## 15. Biến thể

1. **Chu vi phần hợp**: đường quét hai chiều, cộng `|L_sau − L_trước|` cho cạnh dọc; tương tự cho cạnh ngang.
2. **Diện tích bị phủ ít nhất `k` lần**: cây phân đoạn lưu `len[node][t]` = độ dài bị phủ ≥ `t` lần với `t ≤ k`.
3. **Thêm/xoá hình động** và truy vấn diện tích: cây phân đoạn 2D hoặc xử lý ngoại tuyến.
4. **Hình tròn / đa giác**: không còn nén lưới đơn giản — tích phân số hoặc hình học tính toán.
5. **Ba chiều (thể tích hợp hình hộp)**: nén 3 trục, lưới `O(n³)` ô hoặc quét theo `z` với bài 850 ở mỗi lát.

Minh hoạ biến thể 2 (phủ ít nhất `k` lần) bằng lưới nén, đối chiếu ô đơn vị:

In [23]:
def area_at_least_k(rects, k):
    xs = sorted({x for r in rects for x in (r[0], r[2])})
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    xi = {x: i for i, x in enumerate(xs)}
    yi = {y: i for i, y in enumerate(ys)}
    cover = [[0] * len(ys) for _ in range(len(xs))]
    for x1, y1, x2, y2 in rects:
        for i in range(xi[x1], xi[x2]):
            for j in range(yi[y1], yi[y2]):
                cover[i][j] += 1
    return sum((xs[i + 1] - xs[i]) * (ys[j + 1] - ys[j])
               for i in range(len(xs) - 1) for j in range(len(ys) - 1) if cover[i][j] >= k)

def area_at_least_k_brute(rects, k):
    from collections import Counter
    c = Counter((x, y) for x1, y1, x2, y2 in rects for x in range(x1, x2) for y in range(y1, y2))
    return sum(1 for v in c.values() if v >= k)

for seed in range(100):
    r_ = random.Random(seed)
    rects = random_rects(r_.randint(1, 6), 10, r_)
    for k in (1, 2, 3):
        assert area_at_least_k(rects, k) == area_at_least_k_brute(rects, k)
print("phủ >= 2 lần (ví dụ 1):", area_at_least_k(EXAMPLES[0][0], 2))

phủ >= 2 lần (ví dụ 1): 2


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ô đơn vị | tô từng ô 1×1 | `O(diện tích)` | `O(diện tích)` | đối chiếu, toạ độ nhỏ |
| 2. Nén + lưới | ô không đều, đánh dấu | `O(n³)` | `O(n²)` | dễ viết, `n ≤ 100` |
| 3. Quét + hợp khoảng | `L(x)·Δx`, hợp lại mỗi sự kiện | `O(n² log n)` | `O(n)` | đủ cho `n = 200` |
| 4. Quét + cây phân đoạn | cập nhật `±1`, `length[1]` | `O(n log n)` | `O(n)` | **nộp bài**, `n` lớn |
| 5. Bao hàm – loại trừ | tổng có dấu các giao | `O(2ⁿ·n)` | `O(1)` | `n ≤ 15`, kiểm tra |

## 17. Bài tập

**Bài 1 (đoán output).** `[[0,0,2,2],[2,0,4,2]]`? `[[0,0,4,4],[1,1,3,3],[2,2,5,5]]`?

**Bài 2 (tính tay).** Dựng bảng đường quét cho `[[0,0,3,1],[1,0,2,3],[0,2,3,3]]`.

**Bài 3 (điền TODO).** Viết `union_perimeter(rects)` tính chu vi phần hợp bằng lưới nén (đếm cạnh giữa ô phủ và ô không phủ).

**Bài 4 (tự viết lại).** Viết lại `Solution` với cây phân đoạn dạng **mảng lặp** (không đệ quy) hoặc giải thích vì sao độ sâu đệ quy `O(log n)` là an toàn.

**Bài 5 (mini challenge).** Biến thể 2 dùng **cây phân đoạn**: diện tích bị phủ ít nhất 2 lần trong `O(n log n)`.

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def union_perimeter_todo(rects):
    # TODO: lưới nén covered[i][j]; với mỗi cặp ô kề (hoặc ô biên), nếu khác trạng thái thì cộng độ dài cạnh chung
    pass

# Bài 5 — TODO
def area_at_least_two_segtree_todo(rects):
    # TODO: mỗi nút lưu len1 (phủ >= 1) và len2 (phủ >= 2), tính từ cnt[node] và con
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: hai hình chạm cạnh, không chồng.
- Mức 2: hình 2 nằm trong hình 1; hình 3 chồng góc `[2,4)×[2,4)` với hình 1.
- Mức 3: `8`; `16 + 9 − 4 = 21`.

**Bài 2.**
- Mức 1: sự kiện tại `x = 0, 1, 2, 3`.
- Mức 2: `[0,1)`: phủ `[0,1) ∪ [2,3)` = 2; `[1,2)`: thêm `[0,3)` → 3.
- Mức 3: `2 + 3 + 2 = 7`.

**Bài 3.**
- Mức 1: chu vi = tổng độ dài các cạnh ngăn ô phủ với ô không phủ (hoặc với bên ngoài).
- Mức 2: cạnh dọc giữa cột `i−1` và `i` tại hàng `j` có độ dài `ys[j+1] − ys[j]`.
- Mức 3: thêm một hàng/cột "ngoài" không phủ ở mỗi biên.

**Bài 4.**
- Mức 1: số lá `m ≤ 400` → độ sâu ≈ 9.
- Mức 2: đệ quy an toàn.
- Mức 3: bản lặp: tính từ dưới lên theo chỉ số nút — tuỳ chọn.

**Bài 5.**
- Mức 1: `cnt >= 2` → `len2 = cả đoạn`.
- Mức 2: `cnt == 1` → `len2 = len1 của hai con` (mỗi điểm đã phủ ≥1 ở con thì thêm 1 lần từ nút này).
- Mức 3: `cnt == 0` → `len2 = len2 hai con`; lá với `cnt < 2` và không có con → 0 (hoặc cả đoạn nếu `cnt >= 2`).

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().rectangleArea([[0, 0, 2, 2], [2, 0, 4, 2]]) == 8
assert Solution().rectangleArea([[0, 0, 4, 4], [1, 1, 3, 3], [2, 2, 5, 5]]) == 21

# Bài 2
assert Solution().rectangleArea([[0, 0, 3, 1], [1, 0, 2, 3], [0, 2, 3, 3]]) == 7

# Bài 3
def union_perimeter(rects):
    rects = [r for r in rects if r[0] < r[2] and r[1] < r[3]]
    if not rects:
        return 0
    xs = sorted({x for r in rects for x in (r[0], r[2])})
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    xi = {x: i for i, x in enumerate(xs)}; yi = {y: i for i, y in enumerate(ys)}
    W, H = len(xs) - 1, len(ys) - 1
    cov = [[False] * H for _ in range(W)]
    for x1, y1, x2, y2 in rects:
        for i in range(xi[x1], xi[x2]):
            for j in range(yi[y1], yi[y2]):
                cov[i][j] = True
    def c(i, j):
        return 0 <= i < W and 0 <= j < H and cov[i][j]
    per = 0
    for i in range(W + 1):                       # cạnh dọc tại xs[i]
        for j in range(H):
            if c(i - 1, j) != c(i, j):
                per += ys[j + 1] - ys[j]
    for j in range(H + 1):                       # cạnh ngang tại ys[j]
        for i in range(W):
            if c(i, j - 1) != c(i, j):
                per += xs[i + 1] - xs[i]
    return per

def union_perimeter_brute(rects):
    cells = {(x, y) for x1, y1, x2, y2 in rects for x in range(x1, x2) for y in range(y1, y2)}
    return sum((nx, ny) not in cells for x, y in cells for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)))

for seed in range(150):
    r_ = random.Random(seed)
    rects = random_rects(r_.randint(1, 5), 9, r_)
    assert union_perimeter(rects) == union_perimeter_brute(rects)
print("chu vi ví dụ 1:", union_perimeter(EXAMPLES[0][0]))

# Bài 5
def area_at_least_two_segtree(rects):
    rects = [r for r in rects if r[0] < r[2] and r[1] < r[3]]
    if not rects:
        return 0
    ys = sorted({y for r in rects for y in (r[1], r[3])})
    yi = {y: i for i, y in enumerate(ys)}
    m = len(ys) - 1
    cnt, len1, len2 = [0] * (4 * m), [0] * (4 * m), [0] * (4 * m)
    def pull(node, lo, hi):
        full = ys[hi] - ys[lo]
        leaf = hi - lo == 1
        c1 = 0 if leaf else len1[2 * node] + len1[2 * node + 1]
        c2 = 0 if leaf else len2[2 * node] + len2[2 * node + 1]
        if cnt[node] >= 2:
            len1[node] = len2[node] = full
        elif cnt[node] == 1:
            len1[node] = full
            len2[node] = c1
        else:
            len1[node], len2[node] = c1, c2
    def update(node, lo, hi, l, r, v):
        if r <= lo or hi <= l:
            return
        if l <= lo and hi <= r:
            cnt[node] += v
        else:
            mid = (lo + hi) // 2
            update(2 * node, lo, mid, l, r, v); update(2 * node + 1, mid, hi, l, r, v)
        pull(node, lo, hi)
    ev = sorted([(r[0], 1, yi[r[1]], yi[r[3]]) for r in rects] + [(r[2], -1, yi[r[1]], yi[r[3]]) for r in rects])
    total, px = 0, ev[0][0]
    for x, v, l, r in ev:
        total += len2[1] * (x - px)
        update(1, 0, m, l, r, v)
        px = x
    return total

for seed in range(200):
    r_ = random.Random(seed)
    rects = random_rects(r_.randint(1, 7), 12, r_)
    assert area_at_least_two_segtree(rects) == area_at_least_k(rects, 2)
print("Tất cả đáp án đúng")

chu vi ví dụ 1: 12
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (quét + cây phân đoạn đếm phủ):**

```python
ys = sorted({y for r in R for y in (r[1], r[3])}); yi = {y: i for i, y in enumerate(ys)}; m = len(ys) - 1
cnt = [0]*(4*m); length = [0]*(4*m)
def update(node, lo, hi, l, r, v):
    if r <= lo or hi <= l: return
    if l <= lo and hi <= r: cnt[node] += v
    else:
        mid = (lo + hi)//2; update(2*node, lo, mid, l, r, v); update(2*node+1, mid, hi, l, r, v)
    length[node] = ys[hi]-ys[lo] if cnt[node] else (0 if hi-lo == 1 else length[2*node]+length[2*node+1])
events = sorted((x1, 1, yi[y1], yi[y2]) ... + (x2, -1, yi[y1], yi[y2]) ...)
for x, v, l, r in events:
    total += length[1] * (x - prev_x); update(1, 0, m, l, r, v); prev_x = x
return total % MOD
```

**Checklist trước khi submit:**

- [ ] Tính **hợp**, không cộng diện tích từng hình.
- [ ] Lá cây phân đoạn là **đoạn** `[ys[i], ys[i+1])`, cập nhật nửa mở.
- [ ] Cộng diện tích dải **trước** khi áp sự kiện tại `x`.
- [ ] Bỏ qua (hoặc xử lý an toàn) hình diện tích 0; danh sách sự kiện rỗng → 0.
- [ ] Modulo ở cuối (hoặc chỉ cho phép cộng/nhân), không trộn với so sánh.